<a href="https://colab.research.google.com/github/adityakaldhone09/Pragyan/blob/main/ai_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Summary**

This notebook sets up a system to recommend or find similar job postings based on their descriptions, skills, and titles. Here's a summary of the process:

Data Loading and Preprocessing: It starts by loading job data from an Excel file, which includes 'Job Title', 'Skills', and 'Job Description'. Missing values are handled, and all text fields are combined into a single combined_text column. This combined text is then cleaned by converting it to lowercase and removing special characters.

Feature Extraction (Embeddings):
Sentence Embeddings (Semantic Similarity): A SentenceTransformer model ('all-MiniLM-L6-v2') is used to convert the combined_text into numerical vectors (embeddings).
These embeddings capture the semantic meaning of the job descriptions, allowing for the identification of jobs that are conceptually similar, even if they don't share exact keywords.

TF-IDF (Keyword Importance): A TfidfVectorizer is applied to the combined_text to calculate Term Frequency-Inverse Document Frequency scores. This helps to identify important keywords in each job description and can be used for keyword-based similarity or search.

Model Persistence: Finally, the trained SentenceTransformer model, the generated job embeddings, the TfidfVectorizer, the TF-IDF matrix, and the processed DataFrame are all saved as .pkl files. This allows these components to be easily loaded and reused later without needing to rerun the entire processing pipeline, for example, to find similar jobs for a new query or user profile.


In [ ]:
#!pip install sentence-transformers scikit-learn openpyxl joblib

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Job_Title_Datasets(NEW).xlsx to Job_Title_Datasets(NEW).xlsx


In [ ]:
file_name = list(uploaded.keys())[0]

df = pd.read_excel("/content/job_dataset.xlsx")

df.head()

,Job Title,Skills,Job Description
0,Django Developer,"Python, Django, Flask, RESTful APIs, SQL, Post...",About the company:\nTech Driven Basic (TDB) is...
1,Database Administrator,"Database Administration, SQL, Oracle, MySQL, P...",job detail description position title database...
2,Database Administrator,"Database Administration, SQL, Oracle, MySQL, P...",game changer enterprising entrepreneurial coll...
3,Blockchain Developer,"Solidity, Ethereum, Smart Contracts, Web3.js, ...",We are seeking a Blockchain Developer to build...
4,Wordpress Developer,"WordPress, PHP, MySQL, HTML5, CSS3, JavaScript...",Job Description\nDo you want to join one of th...


In [ ]:
df = df.fillna("")

df.columns = df.columns.str.strip()

print(df.columns)

Index(['Job Title', 'Skills', 'Job Description'], dtype='object')


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text

In [ ]:
df["combined_text"] = (
    df["Job Title"].astype(str) + " " +
    df["Skills"].astype(str) + " " +
    df["Job Description"].astype(str)
)

df["combined_text"] = df["combined_text"].apply(clean_text)

df.head()

,Job Title,Skills,Job Description,combined_text
0,Django Developer,"Python, Django, Flask, RESTful APIs, SQL, Post...",About the company:\nTech Driven Basic (TDB) is...,django developer python django flask restful a...
1,Database Administrator,"Database Administration, SQL, Oracle, MySQL, P...",job detail description position title database...,database administrator database administration...
2,Database Administrator,"Database Administration, SQL, Oracle, MySQL, P...",game changer enterprising entrepreneurial coll...,database administrator database administration...
3,Blockchain Developer,"Solidity, Ethereum, Smart Contracts, Web3.js, ...",We are seeking a Blockchain Developer to build...,blockchain developer solidity ethereum smart c...
4,Wordpress Developer,"WordPress, PHP, MySQL, HTML5, CSS3, JavaScript...",Job Description\nDo you want to join one of th...,wordpress developer wordpress php mysql html5 ...


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
job_embeddings = model.encode(
    df["combined_text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

tfidf_matrix = tfidf.fit_transform(
    df["combined_text"]
)

In [ ]:
joblib.dump(model, "sentence_model.pkl")

joblib.dump(job_embeddings, "job_embeddings.pkl")

joblib.dump(tfidf, "tfidf.pkl")

joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")

joblib.dump(df, "jobs_dataset.pkl")

print("All models saved successfully")

All models saved successfully


In [ ]:
from google.colab import files

files.download("job_embeddings.pkl")
files.download("tfidf.pkl")
files.download("tfidf_matrix.pkl")
files.download("jobs_dataset.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>